# Ad Campaign Performance Analysis
Analysis of ad campaign performance for GrowEasy Analytics, a US SaaS company, using cleaned data (~950 rows, January 2024). The dataset tracks daily ad metrics across campaigns (SpringPromo, WinterSale, ProductLaunch) and platforms (Google, Meta, LinkedIn). This notebook addresses marketing team requests to optimize ad spend and campaign strategy.

**Requests**:
1. Campaign Performance Overview: Summarize metrics and flag inefficient campaigns.
2. Platform Efficiency Analysis: Compare platforms and recommend budget shifts.
3. Temporal Trends and Seasonality: Analyze daily/weekly trends and day-of-week effects.
4. Campaign-Platform Interaction: Evaluate campaign performance by platform.
5. Budget Optimization Recommendations: Propose budget reallocation based on performance.

**Dataset**: `cleaned_ad_campaign_performance.csv`

In [5]:
# Imports and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

np.random.seed(42)  # Reproducibility
sns.set_style('whitegrid')  # Clear plots
df = pd.read_csv('cleaned_ad_campaign_performance.csv')
df['Date'] = pd.to_datetime(df['Date'])  # Ensure datetime
print("Dataset Overview:")
print(df.info())
print("\nSample Data:")
print(df.head())

Dataset Overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 970 entries, 0 to 969
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   AdID          970 non-null    object        
 1   CampaignName  970 non-null    object        
 2   Platform      970 non-null    object        
 3   Date          970 non-null    datetime64[ns]
 4   Impressions   970 non-null    int64         
 5   Clicks        970 non-null    int64         
 6   Cost          970 non-null    float64       
 7   Conversions   970 non-null    int64         
 8   CTR           970 non-null    float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(3)
memory usage: 68.3+ KB
None

Sample Data:
     AdID   CampaignName  Platform       Date  Impressions  Clicks  \
0  AD1001    SpringPromo    Google 2024-01-07         1080      89   
1  AD1002     WinterSale      Meta 2024-01-29          537       7   
2  AD1003    Spri

## Request 1: Campaign Performance Overview
Summarize total `Impressions`, `Clicks`, `Cost`, `Conversions`, average `CTR`, CPC, CPCv, and CR by `CampaignName`. Flag campaigns with CPCv > $50.

In [7]:
# Request 1: Campaign Performance Overview
campaign_summary = df.groupby('CampaignName').agg({
    'Impressions': 'sum',
    'Clicks': 'sum',
    'Cost': 'sum',
    'Conversions': 'sum',
    'CTR': 'mean'
}).round(2)
campaign_summary['CPC'] = (campaign_summary['Cost'] / campaign_summary['Clicks']).replace([np.inf, np.nan], 'N/A')
campaign_summary['CPCv'] = (campaign_summary['Cost'] / campaign_summary['Conversions']).replace([np.inf, np.nan], 'N/A')
campaign_summary['CR'] = (campaign_summary['Conversions'] / campaign_summary['Clicks']).replace([np.inf, np.nan], 'N/A')
campaign_summary['HighCPCv'] = campaign_summary['CPCv'].apply(lambda x: x > 50 if isinstance(x, (int, float)) else False)
print("Campaign Performance Summary:")
print(campaign_summary)

Campaign Performance Summary:
               Impressions  Clicks      Cost  Conversions   CTR       CPC  \
CampaignName                                                                
ProductLaunch       876330   43613  83909.66         2739  0.04  1.923960   
SpringPromo         879230   44949  88710.41         3057  0.05  1.973579   
WinterSale          908637   46815  89029.28         2993  0.05  1.901726   

                    CPCv        CR  HighCPCv  
CampaignName                                  
ProductLaunch  30.635144  0.062802     False  
SpringPromo    29.018780  0.068010     False  
WinterSale     29.745834  0.063933     False  


## Request 2: Platform Efficiency Analysis
Compare performance across platforms (`Google`, `Meta`, `LinkedIn`) to recommend allocation for the remaining $50,000 Q1 budget. Provide total `Cost`, total `Conversions`, average `CTR`, Cost per Conversion (`CPCv`), and Conversion Rate (`CR`) by `Platform`. Flag platforms with <10% of total `Impressions` for under-exposure. Recommend a platform for budget increase based on high `CR`, low `CPCv`, and adequate impression share. Export results to Excel for stakeholder review.

In [9]:
# Assuming df is the cleaned Ad Campaign Performance Dataset with required columns
# Summarize metrics by Platform
platform_summary = df.groupby('Platform').agg({
    'Cost': 'sum',
    'Conversions': 'sum',
    'CTR': 'mean',
    'Impressions': 'sum',
    'Clicks': 'sum'
}).round(2)

# Compute efficiency metrics (pandas handles zero divisions as np.nan)
platform_summary['CPCv'] = platform_summary['Cost'] / platform_summary['Conversions']
platform_summary['CR'] = platform_summary['Conversions'] / platform_summary['Clicks']

# Calculate impression share and flag low impressions
platform_summary['ImpressionShare'] = platform_summary['Impressions'] / platform_summary['Impressions'].sum()
platform_summary['LowImpressions'] = platform_summary['ImpressionShare'] < 0.1

# Recommendation: Prioritize highest CR, lowest CPCv among valid platforms
valid_platforms = platform_summary[platform_summary['CPCv'].notna()]

if valid_platforms.empty:
    recommendation = "No platforms with conversions available."
else:
    # Find the maximum CR value
    max_cr = valid_platforms['CR'].max()
    
    # Select all platforms with the maximum CR
    top_cr_platforms = valid_platforms[valid_platforms['CR'] == max_cr]
    
    # Among these, pick the platform with the lowest CPCv
    best_platform = top_cr_platforms['CPCv'].idxmin()
    
    recommendation = (f"Prioritize {best_platform} for budget increase (highest CR: "
                      f"{top_cr_platforms.loc[best_platform, 'CR']:.2%}, CPCv: "
                      f"${top_cr_platforms.loc[best_platform, 'CPCv']:.2f}), "
                      f"with ImpressionShare: {platform_summary.loc[best_platform, 'ImpressionShare']:.2%}.")

# Print the summary for comparison
print("Platform Efficiency Summary:")
print(platform_summary)

# Print the final recommendation
print(f"\nRecommendation: {recommendation}")

# Export to Excel for stakeholders
platform_summary.to_excel('platform_efficiency_summary.xlsx', index=True)

Platform Efficiency Summary:
              Cost  Conversions   CTR  Impressions  Clicks       CPCv  \
Platform                                                                
Google    89119.08         3164  0.04       933274   46417  28.166587   
LinkedIn  78666.64         2649  0.05       803462   42948  29.696731   
Meta      93863.63         2976  0.04       927461   46012  31.540198   

                CR  ImpressionShare  LowImpressions  
Platform                                             
Google    0.068165         0.350302           False  
LinkedIn  0.061679         0.301578           False  
Meta      0.064679         0.348120           False  

Recommendation: Prioritize Google for budget increase (highest CR: 6.82%, CPCv: $28.17), with ImpressionShare: 35.03%.


## Request 3: Temporal Trends and Seasonality
Analyze daily and weekly ad performance in January 2024 to optimize Q2 ad scheduling. Provide total `Conversions` and average `CTR` by `Date` (daily) and week (using `Date.dt.isocalendar().week`). Identify the top 3 days with the highest `Conversions`. Compare weekday vs. weekend performance (average `Conversions` and `CTR`). Recommend focusing ad spend on weekdays or weekends based on performance. Export results to Excel for stakeholder review.

In [19]:
# Assuming df is the cleaned Ad Campaign Performance Dataset
# Daily trends: Total Conversions and average CTR by Date
daily_trends = df.groupby(df['Date'].dt.date).agg({
    'Conversions': 'sum',
    'CTR': 'mean'
}).round(2)

# Weekly trends: Total Conversions and average CTR by week
weekly_trends = df.groupby(df['Date'].dt.isocalendar().week).agg({
    'Conversions': 'sum',
    'CTR': 'mean'
}).round(2)

# Top 3 days by Conversions
top_days = daily_trends['Conversions'].nlargest(3)

# Weekday vs. Weekend comparison
df['IsWeekend'] = df['Date'].dt.dayofweek >= 5
weekday_weekend_stats = df.groupby('IsWeekend').agg({
    'Conversions': 'mean',
    'CTR': 'mean'
}).round(2)
weekday_weekend_stats.index = ['Weekday', 'Weekend']

# Recommendation: Compare weekday vs. weekend Conversions
if weekday_weekend_stats.loc['Weekday', 'Conversions'] > weekday_weekend_stats.loc['Weekend', 'Conversions']:
    recommendation = "Focus ad spend on weekdays (higher average Conversions)."
else:
    recommendation = "Focus ad spend on weekends (higher average Conversions)."

# Print results for technical comparison
print("Daily Trends (Conversions and CTR):")
print(daily_trends)
print("\nWeekly Trends (Conversions and CTR):")
print(weekly_trends)
print("\nTop 3 Days by Conversions:")
print(top_days)
print("\nWeekday vs. Weekend Performance:")
print(weekday_weekend_stats)
print(f"\nRecommendation: {recommendation}")

# Export to Excel for stakeholders
daily_trends.to_excel('daily_trends.xlsx', index=True)
weekly_trends.to_excel('weekly_trends.xlsx', index=True)
weekday_weekend_stats.to_excel('weekday_weekend_stats.xlsx', index=True)

Daily Trends (Conversions and CTR):
            Conversions   CTR
Date                         
2024-01-01          253  0.05
2024-01-02          260  0.04
2024-01-03          271  0.06
2024-01-04          213  0.05
2024-01-05          371  0.04
2024-01-06          315  0.05
2024-01-07          240  0.05
2024-01-08          204  0.05
2024-01-09          295  0.05
2024-01-10          327  0.05
2024-01-11          348  0.05
2024-01-12          228  0.04
2024-01-13          729  0.04
2024-01-14          231  0.05
2024-01-15          300  0.05
2024-01-16          273  0.04
2024-01-17          323  0.05
2024-01-18          189  0.04
2024-01-19          323  0.04
2024-01-20          189  0.04
2024-01-21          211  0.04
2024-01-22          290  0.05
2024-01-23          244  0.04
2024-01-24          476  0.05
2024-01-25          376  0.05
2024-01-26          205  0.04
2024-01-27          209  0.04
2024-01-28          172  0.05
2024-01-29          215  0.06
2024-01-30          231  0.05
2024

## Request 4: Campaign-Platform Interaction
Analyze performance of campaign-platform combinations (`CampaignName` and `Platform`) to guide Q2 creative strategy. Provide average `CTR` and total `Conversions` by `CampaignName` and `Platform`. Highlight combinations with `CTR` > 0.05 or `Conversions` > 100 as high-performing, and flag those with `CTR` < 0.01 as low-performing, suggesting reasons (e.g., low `Impressions`). Recommend campaign-platform pairs for creative focus based on high `CTR` and `Conversions`. Export results to Excel for stakeholder review.

In [25]:
# Assuming df is the cleaned Ad Campaign Performance Dataset
# Group by CampaignName and Platform, aggregate CTR (mean) and Conversions (sum)
summary = df.groupby(['CampaignName', 'Platform']).agg({
    'CTR': 'mean',
    'Conversions': 'sum',
    'Impressions': 'sum'  # For low-performance analysis
}).reset_index().round(3)

# Filter high-performing combinations (CTR > 0.05 or Conversions > 100)
high_performers = summary[
    (summary['CTR'] > 0.05) | (summary['Conversions'] > 100)
][['CampaignName', 'Platform', 'CTR', 'Conversions']]

# Filter low-performing combinations (CTR < 0.01)
low_performers = summary[
    summary['CTR'] < 0.01
][['CampaignName', 'Platform', 'CTR', 'Conversions']]

# Identify low-impression combinations (bottom 10% of Impressions)
low_impressions = summary[
    summary['Impressions'] < summary['Impressions'].quantile(0.1)
][['CampaignName', 'Platform', 'Impressions']]

# Recommendation: Prioritize high-performing pair with highest CTR
if high_performers.empty:
    recommendation = "No high-performing campaign-platform combinations found."
else:
    top_pair = high_performers.loc[
        high_performers['CTR'].idxmax(), ['CampaignName', 'Platform']
    ]
    recommendation = (f"Prioritize creative development for {top_pair['CampaignName']} "
                     f"on {top_pair['Platform']} (highest CTR: {high_performers['CTR'].max():.2%}, "
                     f"Conversions: {high_performers.loc[high_performers['CTR'].idxmax(), 'Conversions']}).")

# Create pivot table for stakeholder-friendly Excel output
pivot_summary = summary.pivot_table(
    values=['CTR', 'Conversions'],
    index='CampaignName',
    columns='Platform',
    aggfunc='mean',  # Use mean to match summary for CTR
    fill_value=0
).round(3)

# Print results for technical comparison
print("Campaign-Platform Performance (Long Format):")
print(summary[['CampaignName', 'Platform', 'CTR', 'Conversions']])
print("\nHigh-Performing Combinations (CTR > 0.05 or Conversions > 100):")
print(high_performers if not high_performers.empty else "None")
print("\nLow-Performing Combinations (CTR < 0.01):")
print(low_performers if not low_performers.empty else "None")
print("\nPotential Reasons for Low Performance (Low Impressions):")
print(low_impressions if not low_impressions.empty else "None")
print(f"\nRecommendation: {recommendation}")

# Export pivot table to Excel for stakeholders
pivot_summary.to_excel('campaign_platform_summary.xlsx', index=True)

Campaign-Platform Performance (Long Format):
    CampaignName  Platform    CTR  Conversions
0  ProductLaunch    Google  0.045          845
1  ProductLaunch  LinkedIn  0.048          961
2  ProductLaunch      Meta  0.040          933
3    SpringPromo    Google  0.045         1225
4    SpringPromo  LinkedIn  0.047          824
5    SpringPromo      Meta  0.048         1008
6     WinterSale    Google  0.045         1094
7     WinterSale  LinkedIn  0.053          864
8     WinterSale      Meta  0.044         1035

High-Performing Combinations (CTR > 0.05 or Conversions > 100):
    CampaignName  Platform    CTR  Conversions
0  ProductLaunch    Google  0.045          845
1  ProductLaunch  LinkedIn  0.048          961
2  ProductLaunch      Meta  0.040          933
3    SpringPromo    Google  0.045         1225
4    SpringPromo  LinkedIn  0.047          824
5    SpringPromo      Meta  0.048         1008
6     WinterSale    Google  0.045         1094
7     WinterSale  LinkedIn  0.053          8

## Request 5: Budget Optimization Recommendations
Analyze campaign-platform combinations (`CampaignName` and `Platform`) to recommend Q2 budget reallocation. Compute Conversion Rate (`CR`), Cost per Conversion (`CPCv`), and current budget allocation (% of total `Cost`) by combination. Identify high performers (`CR` > 0.03 and `CPCv` < 30) and low performers (`CR` < 0.01 or `CPCv` > 50). Recommend reallocating 20% of low-performing budget to high performers, avoiding platforms with `ImpressionShare` < 0.1. Export results to Excel for stakeholder review.

In [96]:
performance = df.groupby(['CampaignName', 'Platform']).agg({
    'Cost': 'sum',
    'Conversions': 'sum',
    'Clicks': 'sum',
    'Impressions': 'sum'
}).reset_index().round(2)

# Computing main metrics for analysis (CR, CPCv, Budget Allocation and Impression Share)
performance['CR'] = performance['Conversions'] / performance['Clicks']
performance['CPCv'] = performance['Cost'] / performance['Conversions']
performance['BudgetShare'] = performance['Cost'] / performance['Cost'].sum()
performance['ImpressionShare'] = performance['Impressions'] / performance['Impressions'].sum()

# Identifying campaign/platform high and low performers (just a mask to filter other DFs)
high_performers = (performance['CR'] > 0.03) & (performance['CPCv'] < 30) & (performance['ImpressionShare'] >= 0.1)
low_performers = (performance['CR'] < 0.02) | (performance['CPCv'] > 40)

if performance[low_performers].empty:
    # Fallback: Select bottom 25% by CR or top 25% by CPCv (Guarantees some budget for reallocation)
    low_performers = (performance['CR'] < performance['CR'].quantile(0.25)) | \
                    (performance['CPCv'] > performance['CPCv'].quantile(0.75))

# Identifying low performers' budget and the size of reallocation needed
budget_low_performers = performance[low_performers]['BudgetShare'].sum()
budget_reallocation = budget_low_performers * 0.2

Low_tier_performers = performance[low_performers][['CampaignName', 'Platform', 'CR', 'CPCv', 'ImpressionShare']] 
top_tier_performers = performance[high_performers][['CampaignName', 'Platform', 'CR', 'CPCv', 'ImpressionShare']]

if top_tier_performers.empty:
    recommendation = "No high-performing combinations with sufficient impression share."
else:
    top_pair = top_tier_performers.loc[
        top_tier_performers['CR'].idxmax(), ['CampaignName', 'Platform']
    ]
    recommendation = (f"Reallocate {budget_reallocation:.2%} of low-performing budget "
                     f"to {top_pair['CampaignName']} on {top_pair['Platform']} "
                     f"(highest CR: {top_tier_performers['CR'].max():.2%}, "
                     f"CPCv: ${top_tier_performers['CPCv'].min():.2f}).")

# Create pivot table for stakeholder-friendly Excel output
pivot_summary = performance.pivot_table(
    values=['CR', 'CPCv', 'BudgetShare'],
    index='CampaignName',
    columns='Platform',
    aggfunc='mean',
    fill_value=0
).round(3)

# Print results for technical comparison
print("Campaign-Platform Performance:")
print(performance[['CampaignName', 'Platform', 'CR', 'CPCv', 'BudgetShare', 
                  'ImpressionShare']])
print("\nHigh-Performing Combinations (CR > 0.03, CPCv < 30, ImpressionShare >= 0.1):")
print(top_tier_performers if not top_tier_performers.empty else "None")

print("\nLow-Performing Combinations (performance['CR'] < quantile(0.25), (performance['CPCv'] > quantile(0.75):")
print(Low_tier_performers if not Low_tier_performers.empty else "None")
print(f"\nRecommendation: {recommendation}")
print(f'Reallocation Amount: ${(performance['Cost'].sum() * budget_reallocation):.2f}')

# Export pivot table to Excel for stakeholders
pivot_summary.to_excel('budget_optimization_summary.xlsx', index=True)

Campaign-Platform Performance:
    CampaignName  Platform        CR       CPCv  BudgetShare  ImpressionShare
0  ProductLaunch    Google  0.059402  31.993491     0.103323         0.102435
1  ProductLaunch  LinkedIn  0.061312  30.383881     0.111596         0.114321
2  ProductLaunch      Meta  0.068033  29.663719     0.105776         0.112172
3    SpringPromo    Google  0.070928  27.882155     0.130540         0.131832
4    SpringPromo  LinkedIn  0.069979  27.991481     0.088152         0.081664
5    SpringPromo      Meta  0.063384  31.239881     0.120351         0.116521
6     WinterSale    Google  0.073319  25.529196     0.106742         0.116035
7     WinterSale  LinkedIn  0.055746  30.558750     0.100909         0.105592
8     WinterSale      Meta  0.063129  33.524232     0.132611         0.119427

High-Performing Combinations (CR > 0.03, CPCv < 30, ImpressionShare >= 0.1):
    CampaignName Platform        CR       CPCv  ImpressionShare
2  ProductLaunch     Meta  0.068033  29.663719 